# Person 1 - Data & Model Engineer (Colab Workflow)

This notebook sets up and validates the Person 1 pipeline:

1. Resume parser (`talent_core/person1/resume_parser.py`)
2. Corpus builder + Word2Vec trainer (`talent_core/person1/word2vec_trainer.py`)
3. Similarity utilities (`talent_core/person1/utils.py`)

## Kaggle datasets to place in `talent_core/person1/data/`

Search and download these exact datasets from kaggle.com:
- Resume Dataset by gauravduttakiit (`UpdatedResumeDataSet.csv`)
- Job Description Dataset by ravindrasinghrana
- Skill Extractor Dataset by jatin2003

After downloading, upload/copy CSVs into `talent_core/person1/data/`.

In [1]:
# Colab/local setup
%pip install -q gensim pdfplumber spacy pandas numpy
!python -m spacy download en_core_web_sm -q

print("Setup complete")

Note: you may need to restart the kernel to use updated packages.
âœ” Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Setup complete


In [2]:
from pathlib import Path

ROOT = Path.cwd()
PERSON1_DIR = ROOT / "talent_core" / "person1"
DATA_DIR = PERSON1_DIR / "data"
MODELS_DIR = PERSON1_DIR / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Person1 dir:", PERSON1_DIR)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)

Person1 dir: c:\PROJECTS\resume-checker\talent_core\person1
Data dir: c:\PROJECTS\resume-checker\talent_core\person1\data
Models dir: c:\PROJECTS\resume-checker\talent_core\person1\models


In [3]:
# Validate required Python files exist
required_files = [
    PERSON1_DIR / "resume_parser.py",
    PERSON1_DIR / "word2vec_trainer.py",
    PERSON1_DIR / "utils.py",
]

for fp in required_files:
    print(f"{fp.name}:", "OK" if fp.exists() else "MISSING")

assert all(fp.exists() for fp in required_files), "One or more required files are missing"
print("All required source files are present.")

resume_parser.py: OK
word2vec_trainer.py: OK
utils.py: OK
All required source files are present.


In [4]:
# Quick parser smoke test (no PDF needed)
import importlib.util

parser_spec = importlib.util.spec_from_file_location("resume_parser", PERSON1_DIR / "resume_parser.py")
resume_parser = importlib.util.module_from_spec(parser_spec)
parser_spec.loader.exec_module(resume_parser)

dummy_text = """
Aarav Menon
SKILLS
Python, FastAPI, Docker, PostgreSQL, AWS, Git
EXPERIENCE
BlueOrbit Systems - Senior Backend Engineer - 4.5 years
PROJECTS
Order API: Python, FastAPI, PostgreSQL, Docker
EDUCATION
B.Tech in Computer Science
National Institute of Technology Trichy
"""

candidate_profile = resume_parser.parse_resume_text(dummy_text)
print(candidate_profile)

# Required shape checks
assert isinstance(candidate_profile.get("name"), str)
assert isinstance(candidate_profile.get("skills"), list)
assert isinstance(candidate_profile.get("projects"), list)
assert isinstance(candidate_profile.get("experience"), list)
assert isinstance(candidate_profile.get("education"), dict)
print("candidate_profile structure check: PASS")

{'name': 'Aarav Menon', 'skills': ['aws', 'docker', 'fastapi', 'git', 'postgresql', 'python', 'sql'], 'projects': [{'name': 'Order API', 'tech': ['docker', 'fastapi', 'postgresql', 'python']}], 'experience': [{'company': 'BlueOrbit Systems', 'role': 'Senior Backend Engineer', 'years': 4.5}], 'education': {'degree': 'B.Tech in Computer Science', 'university': 'Computer Science National Institute of Technology Trichy'}}
candidate_profile structure check: PASS


In [22]:
# Train Word2Vec if Kaggle CSVs are available
import importlib.util

trainer_spec = importlib.util.spec_from_file_location("word2vec_trainer", PERSON1_DIR / "word2vec_trainer.py")
word2vec_trainer = importlib.util.module_from_spec(trainer_spec)
trainer_spec.loader.exec_module(word2vec_trainer)

csv_files = list(DATA_DIR.glob("*.csv"))
print("CSV files found:", [f.name for f in csv_files])

if not csv_files:
    print("No CSVs found in data/. Upload Kaggle datasets, then rerun this cell.")
else:
    tokenized = word2vec_trainer.build_corpus_from_data(DATA_DIR, DATA_DIR / "corpus.txt")
    model = word2vec_trainer.train_word2vec(
        tokenized_sentences=tokenized,
        model_path=MODELS_DIR / "skill_w2v.model",
        vector_size=100,
        window=5,
        min_count=2,
        epochs=10,
    )
    print("Model trained and saved:", MODELS_DIR / "skill_w2v.model")
    print("Vocabulary size:", len(model.wv.key_to_index))

CSV files found: ['benefits.csv', 'companies.csv', 'company_industries.csv', 'company_specialities.csv', 'employee_counts.csv', 'job_industries.csv', 'job_postings.csv', 'job_skills.csv', 'Resume.csv']
Model trained and saved: c:\PROJECTS\resume-checker\talent_core\person1\models\skill_w2v.model
Vocabulary size: 100176


In [23]:
# Validate utils.py APIs (skill_similarity, most_similar_skills, get_skill_vector)
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

model_file = MODELS_DIR / "skill_w2v.model"
if not model_file.exists():
    print("Model file missing. Run training cell after adding datasets.")
else:
    print("skill_similarity('python', 'python'):", utils.skill_similarity("python", "python"))
    print("top similar to 'react':", utils.most_similar_skills("react", topn=5))
    vec = utils.get_skill_vector("python")
    print("vector shape:", vec.shape)

    assert isinstance(utils.skill_similarity("python", "python"), float)
    assert isinstance(utils.most_similar_skills("react", topn=5), list)
    assert hasattr(vec, "shape")
    print("utils API checks: PASS")

skill_similarity('python', 'python'): 1.0
top similar to 'react': [('nodejs', 0.7965), ('typescript', 0.7711), ('javascript', 0.7607), ('angular', 0.7438), ('js', 0.7418)]
vector shape: (100,)
utils API checks: PASS


## Use These Two KaggleHub Datasets

Using the exact datasets you requested:
1. `rajatraj0502/linkedin-job-2023`
2. `hassnainzaidi/resume-classification-dataset-for-nlp`

This cell downloads both and copies discovered CSV files into `talent_core/person1/data/`.

In [7]:
%pip install -q kagglehub

from pathlib import Path
import shutil
import kagglehub

DATA_DIR = Path("talent_core/person1/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download latest versions (exact datasets requested)
linkedin_path = Path(kagglehub.dataset_download("rajatraj0502/linkedin-job-2023"))
resume_path = Path(kagglehub.dataset_download("hassnainzaidi/resume-classification-dataset-for-nlp"))

print("LinkedIn dataset path:", linkedin_path)
print("Resume dataset path:", resume_path)

# Copy all CSV files into Person 1 data directory
copied = []
for root in [linkedin_path, resume_path]:
    for csv in root.rglob("*.csv"):
        target = DATA_DIR / csv.name
        if target.exists():
            stem = target.stem
            suffix = target.suffix
            target = DATA_DIR / f"{stem}_from_{root.name}{suffix}"
        shutil.copy2(csv, target)
        copied.append(target)

print("\nCopied CSV files into", DATA_DIR)
for fp in copied:
    print("-", fp.name)

if not copied:
    print("No CSV files found in downloaded dataset folders.")

Note: you may need to restart the kernel to use updated packages.


c:\PROJECTS\resume-checker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 21.4M/21.4M [00:03<00:00, 6.06MB/s]

Extracting files...


100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 62.5M/62.5M [00:11<00:00, 5.89MB/s]

Extracting files...


LinkedIn dataset path: C:\Users\A S S S Koundinya\.cache\kagglehub\datasets\rajatraj0502\linkedin-job-2023\versions\1
Resume dataset path: C:\Users\A S S S Koundinya\.cache\kagglehub\datasets\hassnainzaidi\resume-classification-dataset-for-nlp\versions\1

Copied CSV files into talent_core\person1\data
- benefits.csv
- companies.csv
- company_industries.csv
- company_specialities.csv
- employee_counts.csv
- job_industries.csv
- job_postings.csv
- job_skills.csv
- Resume.csv


In [16]:
# Requested pairwise similarity tests
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pairs = [
    ("java script", "java_script"),
    ("flask", "djan go"),
    ("flask", "python"),
    ("python", "kubernetes"),
]

print("Pairwise skill similarity results:")
for a, b in pairs:
    print(f"- ({a}, {b}) -> {utils.skill_similarity(a, b)}")

Pairwise skill similarity results:
- (java script, java_script) -> 1.0
- (flask, djan go) -> 0.9807
- (flask, python) -> 0.901
- (python, kubernetes) -> 0.2866


In [17]:
# Additional requested pairwise tests
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pairs = [
    ("python", "docker"),
    ("html", "ml"),
    ("ml", "dl"),
]

print("Additional pairwise skill similarity results:")
for a, b in pairs:
    print(f"- ({a}, {b}) -> {utils.skill_similarity(a, b)}")

Additional pairwise skill similarity results:
- (python, docker) -> 0.2967
- (html, ml) -> 0.6252
- (ml, dl) -> 0.4893


In [18]:
# Quick check: machine learning vs deep learning
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pair = ("machine learning", "deep learning")
print(f"{pair} -> {utils.skill_similarity(pair[0], pair[1])}")

('machine learning', 'deep learning') -> 0.8976


In [19]:
# Quick check: java vs django
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pair = ("java", "django")
print(f"{pair} -> {utils.skill_similarity(pair[0], pair[1])}")

('java', 'django') -> 0.5956


In [20]:
# Quick check: html vs machine learning
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pair = ("html", "machine learning")
print(f"{pair} -> {utils.skill_similarity(pair[0], pair[1])}")

('html', 'machine learning') -> 0.4338


In [21]:
# Abbreviation + typo normalization checks (after utils.py patch)
import importlib.util

utils_spec = importlib.util.spec_from_file_location("utils", PERSON1_DIR / "utils.py")
utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

pairs = [
    ("ml", "dl"),
    ("ml", "machine learning"),
    ("dl", "deep learning"),
    ("machine learnign", "deep learing"),
]

print("Abbreviation/typo similarity results:")
for a, b in pairs:
    print(f"- ({a}, {b}) -> {utils.skill_similarity(a, b)}")

Abbreviation/typo similarity results:
- (ml, dl) -> 0.8976
- (ml, machine learning) -> 1.0
- (dl, deep learning) -> 1.0
- (machine learnign, deep learing) -> 0.8976
